## Extract Frames

In [19]:
import os
import argparse
import cv2
import pandas as pd
import subprocess
from datetime import datetime, timedelta
from glob import glob
import piexif
import json
from tqdm.notebook import tqdm  # For Jupyter progress bars

def extract_frames(video_path, output_folder, frame_interval):
    """
    Extract every Xth frame from a video and save to output folder.
    Append frame data to an existing CSV or create a new one if it doesn't exist.
    
    Args:
        video_path (str): Path to the video file
        output_folder (str): Directory to save extracted frames
        frame_interval (int): Extract every Xth frame
        
    Returns:
        pd.DataFrame: DataFrame containing information about extracted frames
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")
    
    # Open the video file
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        print(f"Could not open video file: {video_path}")
        return pd.DataFrame()  # Return empty DataFrame on error
    
    # Get video properties
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    
    # Get video file information
    video_filename = os.path.basename(video_path)
    video_name = os.path.splitext(video_filename)[0]
    
    print(f"Video: {video_filename}")
    print(f"FPS: {fps:.2f}")
    print(f"Total frames: {frame_count}")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Extracting every {frame_interval}th frame...")
    
    # Process video frames
    current_frame = 0
    saved_count = 0
    
    # Create a list to store frame data for the output DataFrame
    frame_data = []
    
    while True:
        # Read the next frame
        success, frame = video.read()
        if not success:
            break
        
        # Process every Xth frame
        if current_frame % frame_interval == 0:
            # Format frame number with leading zeros based on total frames
            frame_num = str(current_frame).zfill(len(str(frame_count)))
            
            # Calculate time in seconds
            frame_time_seconds = current_frame / fps
            
            # Format time as HH:MM:SS.ms
            time_obj = timedelta(seconds=frame_time_seconds)
            hours, remainder = divmod(time_obj.seconds, 3600)
            minutes, seconds = divmod(remainder, 60)
            milliseconds = int(time_obj.microseconds / 1000)
            
            # Format time string for filename (HHMMSS_ms)
            time_str_filename = f"{hours:02d}{minutes:02d}{seconds:02d}_{milliseconds:03d}"
            
            # Format time string for display (HH:MM:SS.ms)
            time_str_display = f"{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}"
            
            # Create filename with time information
            filename = f"{video_name}_frame{frame_num}_t{time_str_filename}.jpg"
            filepath = os.path.join(output_folder, filename)
            
            # Save the frame
            cv2.imwrite(filepath, frame)
            
            # Store frame info
            frame_info = {
                'filename': filename,
                'filepath': filepath,
                'frame_number': current_frame,
                'frame_time_seconds': frame_time_seconds,
                'frame_time_hhmmss': time_str_display,
                'video_name': video_name,
                'video_fps': fps,
                'video_duration': duration
            }
            
            # Add frame info to our list
            frame_data.append(frame_info)
            
            saved_count += 1
            if saved_count % 20 == 0:
                print(f"Saved {saved_count} frames...")
        
        current_frame += 1
    
    video.release()
    print(f"Extraction complete. Saved {saved_count} frames to {output_folder}")
    
    # Create DataFrame from frame data
    new_frame_df = pd.DataFrame(frame_data)
    
    # Define CSV path
    csv_path = os.path.join(output_folder, "extracted_frames_log.csv")
    
    # Check if CSV exists and append or create new
    if os.path.exists(csv_path):
        # Read existing CSV
        existing_df = pd.read_csv(csv_path)
        # Append new data
        updated_df = pd.concat([existing_df, new_frame_df], ignore_index=True)
        # Save updated DataFrame
        updated_df.to_csv(csv_path, index=False)
        print(f"Appended {len(new_frame_df)} new frame entries to {csv_path}")
    else:
        # Create new CSV
        new_frame_df.to_csv(csv_path, index=False)
        print(f"Created new frame log at {csv_path} with {len(new_frame_df)} entries")
    
    return new_frame_df

def process_video_folder(input_folder, output_folder, frame_interval, video_extensions=['.mp4', '.avi', '.mov', '.mkv']):
    """
    Process all videos in a folder (including subdirectories) and extract frames.
    
    Args:
        input_folder (str): Root folder containing videos
        output_folder (str): Directory to save extracted frames
        frame_interval (int): Extract every Xth frame
        video_extensions (list): List of video file extensions to process (case-insensitive)
        
    Returns:
        int: Number of videos processed
    """
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")
    
    # Find all video files in the directory and subdirectories
    all_files = []
    
    # Walk through directory structure
    for root, _, files in os.walk(input_folder):
        for file in files:
            # Check if file extension matches any of our video extensions (case-insensitive)
            if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                all_files.append(os.path.join(root, file))
    
    # Sort files for consistent processing order
    all_files.sort()
    
    print(f"Found {len(all_files)} video files to process")
    
    # Process each video
    video_count = 0
    for video_path in tqdm(all_files, desc="Processing Videos"):
        # Get relative path to maintain folder structure
        rel_path = os.path.relpath(video_path, input_folder)
        rel_dir = os.path.dirname(rel_path)
        
        # Create output directory that mirrors the input structure
        video_output_folder = os.path.join(output_folder, rel_dir)
        if not os.path.exists(video_output_folder):
            os.makedirs(video_output_folder)
        
        print(f"\nProcessing video {video_count+1}/{len(all_files)}: {rel_path}")
        
        # Extract frames
        try:
            extract_frames(video_path, video_output_folder, frame_interval)
            video_count += 1
        except Exception as e:
            print(f"Error processing {video_path}: {str(e)}")
            continue
    
    print(f"\nAll videos processed. Extracted frames from {video_count}/{len(all_files)} videos.")
    return video_count

In [ ]:
INPUT_FOLDER = "../vids/"
OUTPUT_FOLDER = "video_frames/"
FRAME_INTERVAL = 100

VIDEO_EXTENSIONS = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']

video_count = process_video_folder(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    frame_interval=FRAME_INTERVAL,
    video_extensions=VIDEO_EXTENSIONS
)

print(f"Processed {video_count} videos in total.")

# Display a sample of the extracted frames log
csv_path = os.path.join(OUTPUT_FOLDER, "extracted_frames_log.csv")
if os.path.exists(csv_path):
    log_df = pd.read_csv(csv_path)
    print(f"\nExtracted Frames Log (sample of {min(5, len(log_df))} entries out of {len(log_df)} total):")
    display(log_df.head())

Found 46 video files to process


Processing Videos:   0%|          | 0/46 [00:00<?, ?it/s]


Processing video 1/46: ._Test_drive_3.MP4
Could not open video file: ../vids/._Test_drive_3.MP4

Processing video 2/46: ._gps_test_1.MP4
Could not open video file: ../vids/._gps_test_1.MP4

Processing video 3/46: ._gps_test_2.MP4
Could not open video file: ../vids/._gps_test_2.MP4

Processing video 4/46: ._gps_test_3.MP4
Could not open video file: ../vids/._gps_test_3.MP4

Processing video 5/46: ._test_drive_4.MP4
Could not open video file: ../vids/._test_drive_4.MP4

Processing video 6/46: ._test_drive_5.mp4
Could not open video file: ../vids/._test_drive_5.mp4

Processing video 7/46: ._test_drive_6.mp4
Could not open video file: ../vids/._test_drive_6.mp4

Processing video 8/46: ._test_drive_7.MP4
Could not open video file: ../vids/._test_drive_7.MP4

Processing video 9/46: ._test_drive_8.mp4
Could not open video file: ../vids/._test_drive_8.mp4

Processing video 10/46: Test_drive_3.MP4
Video: Test_drive_3.MP4
FPS: 29.97
Total frames: 21401
Duration: 714.08 seconds
Extracting every 

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6d67661c0] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._Test_drive_3.MP4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c6b80640] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._gps_test_1.MP4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c69ef880] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._gps_test_2.MP4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c6b80640] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._gps_test_3.MP4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c6b80640] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._test_drive_4.MP4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c69ef880] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._test_drive_5.mp4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c69ef880] moov atom not found
OpenCV: Couldn't read video stream from file "../vids/._test_drive_6.mp4"
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7fc6c69ef880] moov

Saved 20 frames...
Saved 40 frames...
Saved 60 frames...
Saved 80 frames...
Saved 100 frames...
